In [7]:
# 处理PMF输出数据，提取质量浓度并保存为新的CSV文件
# 将前处理, 更改列名提前到这一步，方便后续处理
import pandas as pd

MAM_FACTOR_NAMES = {
    'Factor 1': 'Mineral dust',
    'Factor 2': 'Coal combustion',
    'Factor 3': 'Mixed industrial emissions',
    'Factor 4': 'Secondary formation',
    'Factor 5': 'Vehicle emissions',
}

JJA_FACTOR_NAMES = {
    'Factor 1': 'Secondary nitrate',
    'Factor 2': 'Vehicle emissions',
    'Factor 3': 'Mixed industrial emissions',
    'Factor 4': 'Mineral dust',
    'Factor 5': 'Secondary sulfate',
}

SON_FACTOR_NAMES = {
    'Factor 1': 'Secondary nitrate',
    'Factor 2': 'Mixed industrial emissions',
    'Factor 3': 'Mineral dust',
    'Factor 4': 'Vehicle emissions',
    'Factor 5': 'Power plant',
}

DJF_FACTOR_NAMES = {
    'Factor 1': 'Coal combustion',
    'Factor 2': 'Mineral dust',
    'Factor 3': 'Vehicle emissions',
    'Factor 4': 'Secondary formation',
    'Factor 5': 'Smelting industry',
    'Factor 6': 'Fireworks', 
    'Factor 7': 'Power plant',
}

def load_pmf_data(filepath):
    # 读取PMF数据
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    conc_data = []
    state = 0 # 0: 寻找质量浓度, 1: 读取质量浓度
    headers = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        if "Factor Contributions (conc. units)" in line:
            state = 1
            continue

        if state == 1:
            if line.startswith(",,,,Factor 1"):
                headers = line.split(",")[3:]
                headers[0] = "Time"
                continue

            parts = line.split(",")
            if len(parts) > 2:
                # 时间转换为datetime格式
                time = pd.to_datetime(parts[1], format='%m/%d/%y %H:%M')
                conc_values = [float(x) for x in parts[2:]]
                conc_data.append([time] + conc_values)
    # 创建DataFrame
    df = pd.DataFrame(conc_data, columns=headers)
    return df

def clean_factor_names(df, season):
    factor_cols = df.columns[1:]  # 因子列
    df[factor_cols] = df[factor_cols].clip(lower=0)
    for col in factor_cols:
        if df[col].var() == 0:
            df.drop(columns=[col], inplace=True)
            print(f"{season}: Dropped {col} due to zero variance")
    
    if season == "MAM":
        return df.rename(columns=MAM_FACTOR_NAMES)
    elif season == "JJA":
        df['Factor 1'] = df['Factor 1'] + df['Factor 5']  # 合并二次硫酸盐和二次硝酸盐
        df.drop(columns=['Factor 5'], inplace=True)  # 删除原来的二次硫酸盐列
        df.rename(columns={'Factor 1': 'Secondary formation'}, inplace=True)  # 重命名为二次形成
        return df.rename(columns=JJA_FACTOR_NAMES)
    elif season == "SON":
        df.rename(columns={'Factor 1': 'Secondary formation'}, inplace=True)  # 重命名为二次形成
        return df.rename(columns=SON_FACTOR_NAMES)
    elif season == "DJF":
        return df.rename(columns=DJF_FACTOR_NAMES)
    else:
        raise ValueError("Invalid season")

# 主程序
for season in ["MAM", "JJA", "SON", "DJF"]:
    file_path = rf"E:\Coding\Data\Lanzhou_chemical\PMF\{season}_contributions.csv"
    conc_df = load_pmf_data(file_path)
    conc_df = clean_factor_names(conc_df, season)
    conc_df.to_csv(rf"E:\Coding\Data\Lanzhou_chemical\{season}_contrib_clean.csv", index=False)

MAM: Dropped Factor 3 due to zero variance


In [ ]:
# 注:
# 1. 去除不显著的INP数浓度
# 2. 夏季和秋季的"二次硫酸盐"和"二次硝酸盐"修改/合并为"二次形成"

In [25]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
interested_temp = -30
interested_species = "N_INP(#/L)"
def identity(x):
    if x == "MAM":
        return "Spring"
    elif x == "JJA":
        return "Summer"
    elif x == "SON":
        return "Autumn"
    elif x == "DJF":
        return "Winter"

for season in ["MAM", "JJA", "SON", "DJF"]:
    temp = interested_temp
    df_conc = pd.read_csv(rf"E:\Coding\Data\Lanzhou_chemical\{season}_contrib_clean.csv")
    df_inp = pd.read_csv(r"E:\Coding\Data\Lanzhou_aerosol\SMPS+APS\INP+ns(v1.0.2).csv")
    # df_inp = pd.read_csv(r"E:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.2.csv")

    df_conc['Time'] = pd.to_datetime(df_conc['Time'])
    df_inp['Time'] = pd.to_datetime(df_inp['Time'])
    mask = (df_inp['T_a(degC)'] == temp) & (df_inp['Season'] == identity(season)) & (df_inp['Is_Significant'] == True)
    df_season = df_inp[mask].copy()

    # 合并数据
    merged_df = pd.merge_asof(
        df_season.sort_values("Time").reset_index(drop=True),
        df_conc.sort_values("Time").reset_index(drop=True),
        left_on="Time",
        right_on="Time",
        direction="nearest",
        tolerance=pd.Timedelta("1h")
    )
    # print(f"\nMerged {identity(season)} data: {merged_df.shape[0]} rows before dropping NA")
    
    # 剔除合并后仍有缺失值的行
    factor_cols = df_conc.columns[1:]  # 因子列
    merged_df = merged_df.dropna(subset=[*factor_cols, interested_species])

    # 计算相关系数
    print(f"\n Corr ({identity(season)})")
    print("-" * 30)
    # print(merged_df.shape[0], "rows after merging and dropping NA at specified temperature and season")
    

    for factor in factor_cols:
        x = merged_df[factor]
        y = merged_df[interested_species]
        valid_idx = (x > 0) & (y > 0)
        x = np.log10(x[valid_idx])
        y = np.log10(y[valid_idx])
        if len(x) < 3: 
            print(f"{factor} : Not enough valid data points for correlation")
            continue
        
        # 计算相关系数 r 和 p 值
        r, p_value = pearsonr(x, y)

        # 决定显著性符号
        stars = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
        print(f"{factor: <10} : r = {r: >6.3f}, p = {p_value: >5.3f} {stars}")



 Corr (Spring)
------------------------------
Mineral dust : r =  0.541, p = 0.000 ***
Coal combustion : r = -0.125, p = 0.022 *
Secondary formation : r = -0.525, p = 0.000 ***
Vehicle emissions : r =  0.287, p = 0.000 ***

 Corr (Summer)
------------------------------
Secondary Formation : r =  0.255, p = 0.000 ***
Vehicle emissions : r =  0.189, p = 0.007 **
Mixed industrial emissions : r =  0.137, p = 0.076 
Mineral dust : r =  0.390, p = 0.000 ***

 Corr (Autumn)
------------------------------
Secondary Formation : r = -0.330, p = 0.000 ***
Mixed industrial emissions : r =  0.339, p = 0.000 ***
Mineral dust : r =  0.798, p = 0.000 ***
Vehicle emissions : r = -0.513, p = 0.000 ***
Power plant : r =  0.005, p = 0.960 

 Corr (Winter)
------------------------------
Coal combustion : r =  0.114, p = 0.207 
Mineral dust : r =  0.046, p = 0.598 
Vehicle emissions : r =  0.151, p = 0.079 
Secondary formation : r = -0.213, p = 0.015 *
Smelting industry : r = -0.177, p = 0.046 *
Fireworks 

In [11]:
# 合并四个季节的数据
# 仅保留四季均有的因子，方便后续分析
import pandas as pd

target_factors = ['Mineral dust', 'Vehicle emissions', 'Secondary formation']

df_all = pd.DataFrame()

for season in ["DJF", "MAM", "JJA", "SON"]: # 时间上冬在前面(2024年冬季)
    file_path = rf"E:\Coding\Data\Lanzhou_chemical\{season}_contrib_clean.csv"
    df_season = pd.read_csv(file_path)
    print(df_season.shape[0])
    df_sub = df_season[['Time'] + target_factors]
    df_all = pd.concat([df_all, df_sub], ignore_index=True)
print(f"\nCombined data shape: {df_all.shape[0]} rows, {df_all.shape[1]} columns")
df_all.to_csv(r"E:\Coding\Data\Lanzhou_chemical\source_contrib_clean.csv", index=False)


313
351
89
96

Combined data shape: 849 rows, 4 columns
